# Этап 1 (загрузка)

In [1368]:
import numpy as np
import pandas as pd
import re

df = pd.read_excel('датасетA_check.xlsx', sheet_name = 'Лист2')

In [1369]:
# 2. Переименовывание столбцов в латиницу
df = df.rename(columns={
    'возраст': 'age',
    'отсутствие беременности (год)': 'years_without_pregnancy',
    'метолы ВРТ в анамнезе (кол-во попыток)': 'art_attempts',
    'аллергия (0-нет,1-есть)': 'allergy',
    'ИМТ': 'bmi',
    'беременности,было': 'pregnancies_count',
    'выкидыши': 'miscarriages',
    'роды, было': 'deliveries',
    'менструации с': 'menarche_age',
    'Установились (сразу-1, нет-0)': 'menstruation_established',
    'Регулярные': 'cycle_regular',
    'по кол-во дней': 'cycle_days',
    'через кол-во дней': 'cycle_interval',
    'Объем 1-умеренные 0-скудные 2-обильные': 'bleeding_volume',
    'Болезненные 1-да 0-нет': 'painful_periods',
    'Нарушение менструального цикла': 'cycle_disorder',
    'Половая жизнь': 'sexual_life_start',
    'Супруг, возраст': 'partner_age',
    'Наследственность': 'heredity',
    'заболевания (по серьезности болезни)': 'disease_severity',
    'СПКЯ': 'pcos',
    'наличие миомы': 'myoma',
    'мужской фактор бесплодия (1-да, 0-женское)': 'male_factor',
    'АМГ': 'amh',
    'ФСГ': 'fsh',
    'Получено': 'oocytes_retrieved',
    'перенос эмбриона': 'embryo_transfer',
    'ЭКО(1) / ИКСИ(2)': 'ivf_icsi',
    'Криоконсервация (1-да, 0-нет)': 'cryopreservation',
    'беременность (1-наступила, 0-нет)': 'pregnancy',
    'роды': 'delivery'
})
df.columns

Index(['age', 'years_without_pregnancy', 'art_attempts', 'allergy', 'bmi',
       'pregnancies_count', 'miscarriages', 'deliveries', 'menarche_age',
       'menstruation_established', 'cycle_regular', 'cycle_days',
       'cycle_interval', 'bleeding_volume', 'painful_periods',
       'cycle_disorder', 'sexual_life_start', 'partner_age', 'heredity',
       'disease_severity', 'pcos', 'myoma', 'male_factor', 'amh', 'fsh',
       'oocytes_retrieved', 'embryo_transfer', 'ivf_icsi', 'cryopreservation',
       'pregnancy', 'delivery'],
      dtype='object')

In [1370]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 231 entries, 0 to 230
Data columns (total 31 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   age                       229 non-null    float64
 1   years_without_pregnancy   229 non-null    float64
 2   art_attempts              58 non-null     float64
 3   allergy                   229 non-null    float64
 4   bmi                       229 non-null    object 
 5   pregnancies_count         229 non-null    object 
 6   miscarriages              229 non-null    object 
 7   deliveries                229 non-null    object 
 8   menarche_age              229 non-null    object 
 9   menstruation_established  229 non-null    object 
 10  cycle_regular             229 non-null    object 
 11  cycle_days                229 non-null    object 
 12  cycle_interval            229 non-null    object 
 13  bleeding_volume           229 non-null    object 
 14  painful_pe

In [1371]:
# подсчет общей доли пропусков
total_cells = df.size
missing_cells = df.isnull().sum().sum()
missing_pct = (missing_cells / total_cells) * 100

print(f"Общая доля пропусков: {missing_pct:.2f}%")

Общая доля пропусков: 8.06%


In [1372]:
df.head(20)

,age,years_without_pregnancy,art_attempts,allergy,bmi,pregnancies_count,miscarriages,deliveries,menarche_age,menstruation_established,...,myoma,male_factor,amh,fsh,oocytes_retrieved,embryo_transfer,ivf_icsi,cryopreservation,pregnancy,delivery
0,29.0,2.0,1.0,0.0,-,0,0,0,12,1,...,0.0,1,-,-,12,1.0,2,0,1,1
1,33.0,9.0,9.0,0.0,23.6,2,0,2,15,1,...,0.0,0,-,-,18,1.0,2,0,0,0
2,31.0,2.0,1.0,0.0,20.8,0,0,0,11,1,...,1.0,0,2.19,5.37,11,1.0,1,0,1,1
3,32.0,2.0,2.0,0.0,19.8,0,0,0,14,1,...,0.0,1,3.97,4.95,15,1.0,2,0,1,1
4,33.0,1.0,1.0,0.0,21,2,2,2,14,1,...,0.0,0,-,-,13,1.0,1,0,0,0
5,41.0,6.0,1.0,0.0,22.4,0,0,0,13,1,...,0.0,0,-,-,4,1.0,2,0,0,0
6,31.0,1.0,0.0,0.0,29.8,0,0,0,13,1,...,0.0,0,-,-,13,1.0,1,0,1,0
7,32.0,3.0,2.0,0.0,21.7,2,2,0,12,1,...,1.0,0,-,-,9,1.0,2,0,1,-
8,27.0,2.0,0.0,1.0,25.5,0,0,0,10,1,...,0.0,1,-,-,8,1.0,1,0,1,-
9,41.0,0.5,0.0,0.0,24,0,0,0,12,1,...,1.0,1,-,-,14,1.0,1,0,1,-


In [1373]:
# 3. Функция обработки диапазонов
def parse_range(value):
    if pd.isna(value):
        return np.nan
    s = str(value).strip().replace(' ', '')
    # Ищем диапазон "число-число"
    match = re.search(r'(\d+)-(\d+)', s)
    if match:
        low = float(match.group(1))
        high = float(match.group(2))
        return (low + high) / 2.0
    try:
        return float(s)
    except ValueError:
        return np.nan

In [1374]:
df = df.replace(['-'], np.nan)

C:\Users\alisa\AppData\Local\Temp\ipykernel_17888\2895516718.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df = df.replace(['-'], np.nan)


In [1375]:
df.loc[45: 65, 'cycle_interval']

45    28 - 70
46         29
47         28
48         29
49         28
50        NaN
51    31 - 35
52    26 - 28
53         30
54    28 - 30
55         28
56    25 - 26
57    28 - 30
58         28
59        NaN
60        NaN
61        NaN
62        NaN
63        NaN
64        NaN
65        NaN
Name: cycle_interval, dtype: object

In [1376]:
# Дополнительно: создаём признак научной новизны
df['cycle_range_reported'] = df['cycle_interval'].apply(lambda x: 1 if pd.notna(x) and '-' in str(x) else 0).astype('Int64')

In [1377]:
# 4. Применяем parse_range **ДО** замены на NaN (самое важное изменение!)
range_cols = ['cycle_days', 'cycle_interval', 'sexual_life_start', 'cycle_disorder', 'heredity', 'menstruation_established']

for col in range_cols:
    if col in df.columns:
        df[col] = df[col].apply(parse_range)

In [1378]:
# 5. Теперь безопасно заменяем оставшиеся заглушки
df = df.replace(['', ' ', 'NaN', 'nan', None], np.nan)

# 6. Удаляем полностью пустые строки
df = df.dropna(subset=df.columns.difference(['cycle_range_reported']), thresh=2)
df = df.reset_index(drop=True)

# 7. Приводим все числовые столбцы к float
numeric_cols = ['age', 'years_without_pregnancy', 'art_attempts', 'allergy', 'bmi',
                'pregnancies_count', 'miscarriages', 'deliveries', 'menarche_age',
                'cycle_days', 'cycle_interval', 'bleeding_volume', 'painful_periods',
                'sexual_life_start', 'partner_age', 'disease_severity', 'amh', 'fsh',
                'oocytes_retrieved', 'embryo_transfer', 'ivf_icsi', 'cryopreservation',
                'pregnancy', 'delivery']

for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

In [1379]:
df.loc[df['pregnancy'].isna() & (df['delivery'] == 1), 'pregnancy'] = 1

In [1380]:
# 9. Финальная проверка и сохранение
print("=== ФИНАЛЬНЫЙ РЕЗУЛЬТАТ ПОСЛЕ ИСПРАВЛЕНИЯ ===")
print(df.shape)
print(df.info())

=== ФИНАЛЬНЫЙ РЕЗУЛЬТАТ ПОСЛЕ ИСПРАВЛЕНИЯ ===
(229, 32)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 229 entries, 0 to 228
Data columns (total 32 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   age                       229 non-null    float64
 1   years_without_pregnancy   229 non-null    float64
 2   art_attempts              58 non-null     float64
 3   allergy                   229 non-null    float64
 4   bmi                       218 non-null    float64
 5   pregnancies_count         227 non-null    float64
 6   miscarriages              222 non-null    float64
 7   deliveries                224 non-null    float64
 8   menarche_age              225 non-null    float64
 9   menstruation_established  56 non-null     float64
 10  cycle_regular             58 non-null     float64
 11  cycle_days                55 non-null     float64
 12  cycle_interval            56 non-null     float64
 13  bleeding_

In [1381]:
df.isnull().sum().sort_values(ascending=False)

oocytes_retrieved           189
cycle_days                  174
cycle_interval              173
menstruation_established    173
painful_periods             173
bleeding_volume             173
art_attempts                171
cycle_regular               171
pregnancy                    98
sexual_life_start            56
fsh                          42
amh                          36
delivery                     34
bmi                          11
partner_age                   9
miscarriages                  7
deliveries                    5
menarche_age                  4
heredity                      4
cryopreservation              3
male_factor                   3
pregnancies_count             2
cycle_disorder                2
ivf_icsi                      2
embryo_transfer               0
age                           0
myoma                         0
pcos                          0
disease_severity              0
years_without_pregnancy       0
allergy                       0
cycle_ra

In [1382]:
df[['cycle_days', 'cycle_interval', 'cycle_range_reported']].head(20)

,cycle_days,cycle_interval,cycle_range_reported
0,5.0,28.0,0
1,5.0,28.0,0
2,4.0,28.0,0
3,4.0,24.0,0
4,4.0,28.0,0
5,4.0,27.0,0
6,4.0,27.0,0
7,5.0,28.0,0
8,3.0,28.0,0
9,3.0,25.0,0


In [1383]:
df.loc[45: 65, 'cycle_days']

45    NaN
46    4.0
47    6.0
48    5.0
49    5.0
50    5.5
51    4.0
52    4.5
53    4.5
54    4.0
55    3.0
56    4.5
57    4.0
58    NaN
59    NaN
60    NaN
61    NaN
62    NaN
63    NaN
64    NaN
65    NaN
Name: cycle_days, dtype: float64

In [1384]:
df.loc[45: 65, 'cycle_interval']

45    49.0
46    29.0
47    28.0
48    29.0
49    28.0
50    33.0
51    27.0
52    30.0
53    29.0
54    28.0
55    25.5
56    29.0
57    28.0
58     NaN
59     NaN
60     NaN
61     NaN
62     NaN
63     NaN
64     NaN
65     NaN
Name: cycle_interval, dtype: float64

In [1385]:
df.loc[45: 65, 'cycle_range_reported']

45    1
46    0
47    0
48    0
49    0
50    1
51    1
52    0
53    1
54    0
55    1
56    1
57    0
58    0
59    0
60    0
61    0
62    0
63    0
64    0
65    0
Name: cycle_range_reported, dtype: Int64

In [1386]:
df.to_excel('dataset_cleaned_step1_fixed.xlsx', index=False)

In [1387]:
df.shape

(229, 32)

In [1388]:
total_cells = df.size
missing_cells = df.isnull().sum().sum()
missing_pct = (missing_cells / total_cells) * 100

print(f"Общая доля пропусков: {missing_pct:.2f}%")

Общая доля пропусков: 23.40%


# Этап 2 (Разделение на delivery и pregnancy)

In [1389]:
# 1. Создаём копии
df_main = df.copy()   # для основной модели (delivery)
df_preg = df.copy()   # для дополнительного эксперимента (pregnancy)

In [1390]:
# 3. Удаляем строки, где нет целевой переменной
df_main = df_main.dropna(subset=['delivery']).reset_index(drop=True)
df_preg = df_preg.dropna(subset=['pregnancy']).reset_index(drop=True)

In [1391]:
# 5. Проверяем результат
print("=== df_main (delivery) ===")
print("Shape:", df_main.shape)
print("delivery non-null:", df_main['delivery'].notna().sum())
print("pregnancy non-null:", df_main['pregnancy'].notna().sum())

print("\n=== df_preg (pregnancy) ===")
print("Shape:", df_preg.shape)
print("pregnancy non-null:", df_preg['pregnancy'].notna().sum())

=== df_main (delivery) ===
Shape: (195, 32)
delivery non-null: 195
pregnancy non-null: 102

=== df_preg (pregnancy) ===
Shape: (131, 32)
pregnancy non-null: 131


In [1392]:
# 6. Сохраняем
df_main.to_excel('dataset_main_delivery_v2.xlsx', index=False)
df_preg.to_excel('dataset_pregnancy_v2.xlsx', index=False)

In [1393]:
df_preg.isnull().sum().sort_values(ascending=False).head(20)

oocytes_retrieved           95
cycle_days                  81
menstruation_established    80
painful_periods             80
bleeding_volume             80
cycle_interval              80
art_attempts                78
cycle_regular               78
sexual_life_start           53
fsh                         41
amh                         35
delivery                    29
bmi                         11
partner_age                  7
menarche_age                 4
cryopreservation             3
male_factor                  3
cycle_disorder               2
ivf_icsi                     2
miscarriages                 1
dtype: int64

In [1394]:
df_main.isnull().sum().sort_values(ascending=False).head(20)

oocytes_retrieved           178
cycle_days                  172
menstruation_established    172
painful_periods             172
bleeding_volume             172
cycle_interval              172
art_attempts                171
cycle_regular               171
pregnancy                    93
sexual_life_start            24
fsh                          21
amh                          18
bmi                           4
deliveries                    3
miscarriages                  3
partner_age                   3
menarche_age                  1
cycle_disorder                1
cryopreservation              1
ivf_icsi                      1
dtype: int64

In [1395]:
# Делаем подсчет пар
df_main[['pregnancy', 'delivery']].value_counts(dropna=False)

pregnancy  delivery
NaN        0.0         93
1.0        1.0         87
0.0        0.0         14
1.0        0.0          1
Name: count, dtype: int64

In [1396]:
df_main.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 195 entries, 0 to 194
Data columns (total 32 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   age                       195 non-null    float64
 1   years_without_pregnancy   195 non-null    float64
 2   art_attempts              24 non-null     float64
 3   allergy                   195 non-null    float64
 4   bmi                       191 non-null    float64
 5   pregnancies_count         195 non-null    float64
 6   miscarriages              192 non-null    float64
 7   deliveries                192 non-null    float64
 8   menarche_age              194 non-null    float64
 9   menstruation_established  23 non-null     float64
 10  cycle_regular             24 non-null     float64
 11  cycle_days                23 non-null     float64
 12  cycle_interval            23 non-null     float64
 13  bleeding_volume           23 non-null     float64
 14  painful_pe

# Этап 3 (обработка DELIVERY)

In [1397]:
df_main.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 195 entries, 0 to 194
Data columns (total 32 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   age                       195 non-null    float64
 1   years_without_pregnancy   195 non-null    float64
 2   art_attempts              24 non-null     float64
 3   allergy                   195 non-null    float64
 4   bmi                       191 non-null    float64
 5   pregnancies_count         195 non-null    float64
 6   miscarriages              192 non-null    float64
 7   deliveries                192 non-null    float64
 8   menarche_age              194 non-null    float64
 9   menstruation_established  23 non-null     float64
 10  cycle_regular             24 non-null     float64
 11  cycle_days                23 non-null     float64
 12  cycle_interval            23 non-null     float64
 13  bleeding_volume           23 non-null     float64
 14  painful_pe

In [1398]:
df_main.head()

,age,years_without_pregnancy,art_attempts,allergy,bmi,pregnancies_count,miscarriages,deliveries,menarche_age,menstruation_established,...,male_factor,amh,fsh,oocytes_retrieved,embryo_transfer,ivf_icsi,cryopreservation,pregnancy,delivery,cycle_range_reported
0,29.0,2.0,1.0,0.0,NaN,0.0,0.0,0.0,12.0,1.0,...,1.0,NaN,NaN,12.0,1.0,2.0,0.0,1.0,1.0,0
1,33.0,9.0,9.0,0.0,23.6,2.0,0.0,2.0,15.0,1.0,...,0.0,NaN,NaN,18.0,1.0,2.0,0.0,0.0,0.0,0
2,31.0,2.0,1.0,0.0,20.8,0.0,0.0,0.0,11.0,1.0,...,0.0,2.19,5.37,11.0,1.0,1.0,0.0,1.0,1.0,0
3,32.0,2.0,2.0,0.0,19.8,0.0,0.0,0.0,14.0,1.0,...,1.0,3.97,4.95,15.0,1.0,2.0,0.0,1.0,1.0,0
4,33.0,1.0,1.0,0.0,21.0,2.0,2.0,2.0,14.0,1.0,...,0.0,NaN,NaN,13.0,1.0,1.0,0.0,0.0,0.0,0


In [1399]:
df_main[['allergy', 'pcos', 'myoma', 'male_factor']].isnull().sum().sort_values(ascending=False)

male_factor    1
allergy        0
pcos           0
myoma          0
dtype: int64

In [1400]:
# 1. Простые заполнения (мода / 0)
df_main['male_factor'] = df_main['male_factor'].fillna(0)

In [1401]:
from sklearn.impute import KNNImputer

# 2. KNNImputer для клинически связанных признаков (рекомендуется для медицинских данных)
impute_cols = ['bmi', 'amh', 'fsh', 'cycle_days', 'cycle_interval', 
               'sexual_life_start', 'partner_age', 'disease_severity']

imputer = KNNImputer(n_neighbors=5)
df_main[impute_cols] = imputer.fit_transform(df_main[impute_cols])

In [1402]:
# 3. Feature Engineering (научная новизна)
df_main['age_group'] = pd.cut(df_main['age'], bins=[0, 30, 35, 40, 50], 
                              labels=[0, 1, 2, 3])
df_main['bmi_category'] = pd.cut(df_main['bmi'], bins=[0, 18.5, 25, 30, 100], 
                                 labels=[0, 1, 2, 3])
df_main['amh_fsh_ratio'] = df_main['amh'] / (df_main['fsh'] + 0.001)   # защита от деления на 0
df_main['total_art_attempts_group'] = pd.cut(df_main['art_attempts'], 
                                             bins=[-1, 0, 2, 5, 10], 
                                             labels=[0, 1, 2, 3])
df_main['poor_responder'] = (df_main['oocytes_retrieved'] < 5).astype(int)  # клинический маркер


In [1403]:
# 4. Финальная проверка
print("=== df_main после Этапа 3 ===")
print(df_main.shape)
print(df_main.isnull().sum().sort_values(ascending=False).head(25))

=== df_main после Этапа 3 ===
(195, 37)
oocytes_retrieved           178
menstruation_established    172
bleeding_volume             172
painful_periods             172
art_attempts                171
total_art_attempts_group    171
cycle_regular               171
pregnancy                    93
miscarriages                  3
deliveries                    3
menarche_age                  1
ivf_icsi                      1
cryopreservation              1
cycle_disorder                1
age                           0
embryo_transfer               0
age_group                     0
delivery                      0
cycle_range_reported          0
amh                           0
bmi_category                  0
amh_fsh_ratio                 0
fsh                           0
heredity                      0
male_factor                   0
dtype: int64


In [1404]:
# 5. Сохраняем
df.to_excel('dataset_main_delivery_final.xlsx', index=False)

### Проверка

In [1405]:
df_main.shape

(195, 37)

In [1406]:
# Таблица Feature Engineering (научная новизна)
df_main[['age_group', 'bmi_category', 'amh_fsh_ratio', 'total_art_attempts_group', 'poor_responder']].head()

,age_group,bmi_category,amh_fsh_ratio,total_art_attempts_group,poor_responder
0,0,1,0.764351,1,0
1,1,1,0.636143,3,0
2,1,1,0.407745,1,0
3,1,1,0.801858,1,0
4,1,1,0.280326,1,0


In [1407]:
df_main.isnull().sum().sort_values(ascending=False).head(20)

oocytes_retrieved           178
menstruation_established    172
bleeding_volume             172
painful_periods             172
art_attempts                171
total_art_attempts_group    171
cycle_regular               171
pregnancy                    93
miscarriages                  3
deliveries                    3
menarche_age                  1
ivf_icsi                      1
cryopreservation              1
cycle_disorder                1
age                           0
embryo_transfer               0
age_group                     0
delivery                      0
cycle_range_reported          0
amh                           0
dtype: int64

In [1408]:
df_main.head()

,age,years_without_pregnancy,art_attempts,allergy,bmi,pregnancies_count,miscarriages,deliveries,menarche_age,menstruation_established,...,ivf_icsi,cryopreservation,pregnancy,delivery,cycle_range_reported,age_group,bmi_category,amh_fsh_ratio,total_art_attempts_group,poor_responder
0,29.0,2.0,1.0,0.0,24.8,0.0,0.0,0.0,12.0,1.0,...,2.0,0.0,1.0,1.0,0,0,1,0.764351,1,0
1,33.0,9.0,9.0,0.0,23.6,2.0,0.0,2.0,15.0,1.0,...,2.0,0.0,0.0,0.0,0,1,1,0.636143,3,0
2,31.0,2.0,1.0,0.0,20.8,0.0,0.0,0.0,11.0,1.0,...,1.0,0.0,1.0,1.0,0,1,1,0.407745,1,0
3,32.0,2.0,2.0,0.0,19.8,0.0,0.0,0.0,14.0,1.0,...,2.0,0.0,1.0,1.0,0,1,1,0.801858,1,0
4,33.0,1.0,1.0,0.0,21.0,2.0,2.0,2.0,14.0,1.0,...,1.0,0.0,0.0,0.0,0,1,1,0.280326,1,0


In [1409]:
df_main[['age', 'years_without_pregnancy', 'art_attempts', 'bmi',
                    'pregnancies_count', 'miscarriages', 'deliveries', 'menarche_age',
                    'cycle_days', 'cycle_interval', 'sexual_life_start', 'partner_age',
                    'disease_severity', 'amh', 'fsh', 'embryo_transfer', 'ivf_icsi',
                    'cryopreservation', 'amh_fsh_ratio']].isnull().sum().sort_values(ascending=False).head(20)

art_attempts               171
miscarriages                 3
deliveries                   3
cryopreservation             1
ivf_icsi                     1
menarche_age                 1
age                          0
disease_severity             0
embryo_transfer              0
fsh                          0
amh                          0
cycle_interval               0
partner_age                  0
sexual_life_start            0
years_without_pregnancy      0
cycle_days                   0
pregnancies_count            0
bmi                          0
amh_fsh_ratio                0
dtype: int64

In [1410]:
df_main[['age_group', 'bmi_category', 'total_art_attempts_group',
                        'allergy', 'pcos', 'myoma', 'male_factor', 'heredity',
                        'cycle_disorder', 'cycle_range_reported', 'poor_responder']].isnull().sum().sort_values(ascending=False).head(20)

total_art_attempts_group    171
cycle_disorder                1
age_group                     0
bmi_category                  0
allergy                       0
pcos                          0
myoma                         0
male_factor                   0
heredity                      0
cycle_range_reported          0
poor_responder                0
dtype: int64

In [1411]:
# Быстрое исправление оставшихся NaN
df_main['art_attempts'] = df_main['art_attempts'].fillna(0)                    # клинически оправдано
df_main['miscarriages'] = df_main['miscarriages'].fillna(0)
df_main['deliveries'] = df_main['deliveries'].fillna(0)
df_main['menarche_age'] = df_main['menarche_age'].fillna(df_main['menarche_age'].median())
df_main['cryopreservation'] = df_main['cryopreservation'].fillna(0)                    # клинически оправдано
df_main['ivf_icsi'] = df_main['ivf_icsi'].fillna(0)
df_main['cycle_disorder'] = df_main['cycle_disorder'].fillna(0)               # 0 = нет нарушения

In [1412]:
df_main[['age', 'years_without_pregnancy', 'art_attempts', 'bmi',
                    'pregnancies_count', 'miscarriages', 'deliveries', 'menarche_age',
                    'cycle_days', 'cycle_interval', 'sexual_life_start', 'partner_age',
                    'disease_severity', 'amh', 'fsh', 'embryo_transfer', 'ivf_icsi',
                    'cryopreservation', 'amh_fsh_ratio']].isnull().sum().sort_values(ascending=False).head(20)

age                        0
sexual_life_start          0
cryopreservation           0
ivf_icsi                   0
embryo_transfer            0
fsh                        0
amh                        0
disease_severity           0
partner_age                0
cycle_interval             0
years_without_pregnancy    0
cycle_days                 0
menarche_age               0
deliveries                 0
miscarriages               0
pregnancies_count          0
bmi                        0
art_attempts               0
amh_fsh_ratio              0
dtype: int64

In [1413]:
#  Пересоздаём группировку после заполнения
df_main['total_art_attempts_group'] = pd.cut(df_main['art_attempts'], 
                                        bins=[-1, 0, 2, 5, 10], 
                                        labels=[0, 1, 2, 3])

In [1414]:
df_main[['age_group', 'bmi_category', 'total_art_attempts_group',
                        'allergy', 'pcos', 'myoma', 'male_factor', 'heredity',
                        'cycle_disorder', 'cycle_range_reported', 'poor_responder']].isnull().sum().sort_values(ascending=False).head(20)

age_group                   0
bmi_category                0
total_art_attempts_group    0
allergy                     0
pcos                        0
myoma                       0
male_factor                 0
heredity                    0
cycle_disorder              0
cycle_range_reported        0
poor_responder              0
dtype: int64

In [1415]:
df_main[['age_group', 'bmi_category', 'amh_fsh_ratio', 'total_art_attempts_group', 'poor_responder']].head()

,age_group,bmi_category,amh_fsh_ratio,total_art_attempts_group,poor_responder
0,0,1,0.764351,1,0
1,1,1,0.636143,3,0
2,1,1,0.407745,1,0
3,1,1,0.801858,1,0
4,1,1,0.280326,1,0


In [1416]:
df_main.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 195 entries, 0 to 194
Data columns (total 37 columns):
 #   Column                    Non-Null Count  Dtype   
---  ------                    --------------  -----   
 0   age                       195 non-null    float64 
 1   years_without_pregnancy   195 non-null    float64 
 2   art_attempts              195 non-null    float64 
 3   allergy                   195 non-null    float64 
 4   bmi                       195 non-null    float64 
 5   pregnancies_count         195 non-null    float64 
 6   miscarriages              195 non-null    float64 
 7   deliveries                195 non-null    float64 
 8   menarche_age              195 non-null    float64 
 9   menstruation_established  23 non-null     float64 
 10  cycle_regular             24 non-null     float64 
 11  cycle_days                195 non-null    float64 
 12  cycle_interval            195 non-null    float64 
 13  bleeding_volume           23 non-null     float64 

In [1417]:
df_main.shape

(195, 37)

In [1418]:
df_main.to_excel('FINAL_delivery.xlsx', index=False)

# Этап 4 (обрабодка PREGNANCY)

In [1419]:
df_preg.head()

,age,years_without_pregnancy,art_attempts,allergy,bmi,pregnancies_count,miscarriages,deliveries,menarche_age,menstruation_established,...,male_factor,amh,fsh,oocytes_retrieved,embryo_transfer,ivf_icsi,cryopreservation,pregnancy,delivery,cycle_range_reported
0,29.0,2.0,1.0,0.0,NaN,0.0,0.0,0.0,12.0,1.0,...,1.0,NaN,NaN,12.0,1.0,2.0,0.0,1.0,1.0,0
1,33.0,9.0,9.0,0.0,23.6,2.0,0.0,2.0,15.0,1.0,...,0.0,NaN,NaN,18.0,1.0,2.0,0.0,0.0,0.0,0
2,31.0,2.0,1.0,0.0,20.8,0.0,0.0,0.0,11.0,1.0,...,0.0,2.19,5.37,11.0,1.0,1.0,0.0,1.0,1.0,0
3,32.0,2.0,2.0,0.0,19.8,0.0,0.0,0.0,14.0,1.0,...,1.0,3.97,4.95,15.0,1.0,2.0,0.0,1.0,1.0,0
4,33.0,1.0,1.0,0.0,21.0,2.0,2.0,2.0,14.0,1.0,...,0.0,NaN,NaN,13.0,1.0,1.0,0.0,0.0,0.0,0


In [1420]:
df_preg.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 131 entries, 0 to 130
Data columns (total 32 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   age                       131 non-null    float64
 1   years_without_pregnancy   131 non-null    float64
 2   art_attempts              53 non-null     float64
 3   allergy                   131 non-null    float64
 4   bmi                       120 non-null    float64
 5   pregnancies_count         131 non-null    float64
 6   miscarriages              130 non-null    float64
 7   deliveries                131 non-null    float64
 8   menarche_age              127 non-null    float64
 9   menstruation_established  51 non-null     float64
 10  cycle_regular             53 non-null     float64
 11  cycle_days                50 non-null     float64
 12  cycle_interval            51 non-null     float64
 13  bleeding_volume           51 non-null     float64
 14  painful_pe

In [1421]:
df_preg.isnull().sum().sort_values(ascending=False).head(25)

oocytes_retrieved           95
cycle_days                  81
menstruation_established    80
painful_periods             80
bleeding_volume             80
cycle_interval              80
art_attempts                78
cycle_regular               78
sexual_life_start           53
fsh                         41
amh                         35
delivery                    29
bmi                         11
partner_age                  7
menarche_age                 4
cryopreservation             3
male_factor                  3
cycle_disorder               2
ivf_icsi                     2
miscarriages                 1
pregnancy                    0
embryo_transfer              0
age                          0
myoma                        0
pcos                         0
dtype: int64

In [1422]:
df_preg[['allergy', 'pcos', 'myoma', 'male_factor']].isnull().sum().sort_values(ascending=False)

male_factor    3
allergy        0
pcos           0
myoma          0
dtype: int64

In [1423]:
df_preg['male_factor'] = df['male_factor'].fillna(0)

In [1424]:
from sklearn.impute import KNNImputer

# 2. KNNImputer для клинически связанных признаков (рекомендуется для медицинских данных)
impute_cols = ['bmi', 'amh', 'fsh', 'cycle_days', 'cycle_interval', 
               'sexual_life_start', 'partner_age', 'disease_severity']

imputer = KNNImputer(n_neighbors=5)
df_preg[impute_cols] = imputer.fit_transform(df_preg[impute_cols])

In [1425]:
# 3. Feature Engineering (научная новизна)
df_preg['age_group'] = pd.cut(df_preg['age'], bins=[0, 30, 35, 40, 50], 
                              labels=[0, 1, 2, 3])
df_preg['bmi_category'] = pd.cut(df_preg['bmi'], bins=[0, 18.5, 25, 30, 100], 
                                 labels=[0, 1, 2, 3])
df_preg['amh_fsh_ratio'] = df_preg['amh'] / (df_preg['fsh'] + 0.001)   # защита от деления на 0
df_preg['total_art_attempts_group'] = pd.cut(df_preg['art_attempts'], 
                                             bins=[-1, 0, 2, 5, 10], 
                                             labels=[0, 1, 2, 3])
df_preg['poor_responder'] = (df_preg['oocytes_retrieved'] < 5).astype(int)  # клинический маркер


In [1426]:
# 4. Финальная проверка
print("=== df_preg после Этапа 1 ===")
print(df_preg.shape)
print(df_preg.isnull().sum().sort_values(ascending=False).head(25))

=== df_preg после Этапа 1 ===
(131, 37)
oocytes_retrieved           95
menstruation_established    80
bleeding_volume             80
painful_periods             80
art_attempts                78
total_art_attempts_group    78
cycle_regular               78
delivery                    29
menarche_age                 4
cryopreservation             3
ivf_icsi                     2
cycle_disorder               2
miscarriages                 1
age                          0
pregnancy                    0
embryo_transfer              0
cycle_range_reported         0
amh                          0
age_group                    0
bmi_category                 0
amh_fsh_ratio                0
fsh                          0
heredity                     0
male_factor                  0
myoma                        0
dtype: int64


In [1427]:
df_preg.shape

(131, 37)

In [1428]:
df_preg[['age_group', 'bmi_category', 'amh_fsh_ratio', 'total_art_attempts_group', 'poor_responder']].head()

,age_group,bmi_category,amh_fsh_ratio,total_art_attempts_group,poor_responder
0,0,1,0.597064,1,0
1,1,1,0.561034,3,0
2,1,1,0.407745,1,0
3,1,1,0.801858,1,0
4,1,1,0.812560,1,0


In [1429]:
df_preg.isnull().sum().sort_values(ascending=False).head(20)

oocytes_retrieved           95
menstruation_established    80
bleeding_volume             80
painful_periods             80
art_attempts                78
total_art_attempts_group    78
cycle_regular               78
delivery                    29
menarche_age                 4
cryopreservation             3
ivf_icsi                     2
cycle_disorder               2
miscarriages                 1
age                          0
pregnancy                    0
embryo_transfer              0
cycle_range_reported         0
amh                          0
age_group                    0
bmi_category                 0
dtype: int64

In [1430]:
df_preg.head()

,age,years_without_pregnancy,art_attempts,allergy,bmi,pregnancies_count,miscarriages,deliveries,menarche_age,menstruation_established,...,ivf_icsi,cryopreservation,pregnancy,delivery,cycle_range_reported,age_group,bmi_category,amh_fsh_ratio,total_art_attempts_group,poor_responder
0,29.0,2.0,1.0,0.0,23.34,0.0,0.0,0.0,12.0,1.0,...,2.0,0.0,1.0,1.0,0,0,1,0.597064,1,0
1,33.0,9.0,9.0,0.0,23.60,2.0,0.0,2.0,15.0,1.0,...,2.0,0.0,0.0,0.0,0,1,1,0.561034,3,0
2,31.0,2.0,1.0,0.0,20.80,0.0,0.0,0.0,11.0,1.0,...,1.0,0.0,1.0,1.0,0,1,1,0.407745,1,0
3,32.0,2.0,2.0,0.0,19.80,0.0,0.0,0.0,14.0,1.0,...,2.0,0.0,1.0,1.0,0,1,1,0.801858,1,0
4,33.0,1.0,1.0,0.0,21.00,2.0,2.0,2.0,14.0,1.0,...,1.0,0.0,0.0,0.0,0,1,1,0.812560,1,0


In [1431]:
df_preg[['age', 'years_without_pregnancy', 'art_attempts', 'bmi',
                    'pregnancies_count', 'miscarriages', 'deliveries', 'menarche_age',
                    'cycle_days', 'cycle_interval', 'sexual_life_start', 'partner_age',
                    'disease_severity', 'amh', 'fsh', 'embryo_transfer', 'ivf_icsi',
                    'cryopreservation', 'amh_fsh_ratio']].isnull().sum().sort_values(ascending=False).head(20)

art_attempts               78
menarche_age                4
cryopreservation            3
ivf_icsi                    2
miscarriages                1
age                         0
partner_age                 0
embryo_transfer             0
fsh                         0
amh                         0
disease_severity            0
cycle_interval              0
sexual_life_start           0
years_without_pregnancy     0
cycle_days                  0
deliveries                  0
pregnancies_count           0
bmi                         0
amh_fsh_ratio               0
dtype: int64

In [1432]:
df_preg[['age_group', 'bmi_category', 'total_art_attempts_group',
                        'allergy', 'pcos', 'myoma', 'male_factor', 'heredity',
                        'cycle_disorder', 'cycle_range_reported', 'poor_responder']].isnull().sum().sort_values(ascending=False).head(20)

total_art_attempts_group    78
cycle_disorder               2
age_group                    0
bmi_category                 0
allergy                      0
pcos                         0
myoma                        0
male_factor                  0
heredity                     0
cycle_range_reported         0
poor_responder               0
dtype: int64

In [1433]:
# Быстрое исправление оставшихся NaN (до препроцессинга!)
df_preg['art_attempts'] = df_preg['art_attempts'].fillna(0)                    # клинически оправдано
df_preg['miscarriages'] = df_preg['miscarriages'].fillna(0)
df_preg['deliveries'] = df_preg['deliveries'].fillna(0)
df_preg['menarche_age'] = df_preg['menarche_age'].fillna(df_preg['menarche_age'].median())
df_preg['cryopreservation'] = df_preg['cryopreservation'].fillna(0)                    # клинически оправдано
df_preg['ivf_icsi'] = df_preg['ivf_icsi'].fillna(0)
df_preg['cycle_disorder'] = df_preg['cycle_disorder'].fillna(0)               # 0 = нет нарушения

In [1434]:
df_preg[['age', 'years_without_pregnancy', 'art_attempts', 'bmi',
                    'pregnancies_count', 'miscarriages', 'deliveries', 'menarche_age',
                    'cycle_days', 'cycle_interval', 'sexual_life_start', 'partner_age',
                    'disease_severity', 'amh', 'fsh', 'embryo_transfer', 'ivf_icsi',
                    'cryopreservation', 'amh_fsh_ratio']].isnull().sum().sort_values(ascending=False).head(20)

age                        0
sexual_life_start          0
cryopreservation           0
ivf_icsi                   0
embryo_transfer            0
fsh                        0
amh                        0
disease_severity           0
partner_age                0
cycle_interval             0
years_without_pregnancy    0
cycle_days                 0
menarche_age               0
deliveries                 0
miscarriages               0
pregnancies_count          0
bmi                        0
art_attempts               0
amh_fsh_ratio              0
dtype: int64

In [1435]:
# 3. Пересоздаём группировку после заполнения
df_preg['total_art_attempts_group'] = pd.cut(df_preg['art_attempts'], 
                                        bins=[-1, 0, 2, 5, 10], 
                                        labels=[0, 1, 2, 3])

In [1436]:
df_preg[['age_group', 'bmi_category', 'total_art_attempts_group',
                        'allergy', 'pcos', 'myoma', 'male_factor', 'heredity',
                        'cycle_disorder', 'cycle_range_reported', 'poor_responder']].isnull().sum().sort_values(ascending=False).head(20)

age_group                   0
bmi_category                0
total_art_attempts_group    0
allergy                     0
pcos                        0
myoma                       0
male_factor                 0
heredity                    0
cycle_disorder              0
cycle_range_reported        0
poor_responder              0
dtype: int64

In [1437]:
df_preg[['age_group', 'bmi_category', 'amh_fsh_ratio', 'total_art_attempts_group', 'poor_responder']].head()

,age_group,bmi_category,amh_fsh_ratio,total_art_attempts_group,poor_responder
0,0,1,0.597064,1,0
1,1,1,0.561034,3,0
2,1,1,0.407745,1,0
3,1,1,0.801858,1,0
4,1,1,0.812560,1,0


In [1438]:
df_preg.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 131 entries, 0 to 130
Data columns (total 37 columns):
 #   Column                    Non-Null Count  Dtype   
---  ------                    --------------  -----   
 0   age                       131 non-null    float64 
 1   years_without_pregnancy   131 non-null    float64 
 2   art_attempts              131 non-null    float64 
 3   allergy                   131 non-null    float64 
 4   bmi                       131 non-null    float64 
 5   pregnancies_count         131 non-null    float64 
 6   miscarriages              131 non-null    float64 
 7   deliveries                131 non-null    float64 
 8   menarche_age              131 non-null    float64 
 9   menstruation_established  51 non-null     float64 
 10  cycle_regular             53 non-null     float64 
 11  cycle_days                131 non-null    float64 
 12  cycle_interval            131 non-null    float64 
 13  bleeding_volume           51 non-null     float64 

In [1439]:
df_preg.to_excel('test_pregnancy.xlsx', index=False)

### Добработка

In [1440]:
df_preg.isnull().sum().sort_values(ascending=False).head(20)

oocytes_retrieved           95
painful_periods             80
bleeding_volume             80
menstruation_established    80
cycle_regular               78
delivery                    29
cryopreservation             0
amh                          0
fsh                          0
embryo_transfer              0
ivf_icsi                     0
age                          0
myoma                        0
pregnancy                    0
cycle_range_reported         0
age_group                    0
bmi_category                 0
amh_fsh_ratio                0
total_art_attempts_group     0
male_factor                  0
dtype: int64

In [1441]:
# 1. Удаляем столбцы с очень большим количеством пропусков (>50%)
cols_to_drop = ['oocytes_retrieved', 'menstruation_established', 
                'bleeding_volume', 'painful_periods', 'cycle_regular']
df_preg = df_preg.drop(columns=cols_to_drop, errors='ignore')

In [1442]:
df_preg.isnull().sum().sort_values(ascending=False).head(20)

delivery                    29
age                          0
years_without_pregnancy      0
total_art_attempts_group     0
amh_fsh_ratio                0
bmi_category                 0
age_group                    0
cycle_range_reported         0
pregnancy                    0
cryopreservation             0
ivf_icsi                     0
embryo_transfer              0
fsh                          0
amh                          0
male_factor                  0
myoma                        0
pcos                         0
disease_severity             0
heredity                     0
partner_age                  0
dtype: int64

In [1443]:
# 3. delivery — заполняем 0 (нет информации о родах)
df_preg['delivery'] = df_preg['delivery'].fillna(0)

In [1444]:
df_preg.isnull().sum().sort_values(ascending=False).head(20)

age                         0
years_without_pregnancy     0
total_art_attempts_group    0
amh_fsh_ratio               0
bmi_category                0
age_group                   0
cycle_range_reported        0
delivery                    0
pregnancy                   0
cryopreservation            0
ivf_icsi                    0
embryo_transfer             0
fsh                         0
amh                         0
male_factor                 0
myoma                       0
pcos                        0
disease_severity            0
heredity                    0
partner_age                 0
dtype: int64

In [1445]:
df_preg.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 131 entries, 0 to 130
Data columns (total 32 columns):
 #   Column                    Non-Null Count  Dtype   
---  ------                    --------------  -----   
 0   age                       131 non-null    float64 
 1   years_without_pregnancy   131 non-null    float64 
 2   art_attempts              131 non-null    float64 
 3   allergy                   131 non-null    float64 
 4   bmi                       131 non-null    float64 
 5   pregnancies_count         131 non-null    float64 
 6   miscarriages              131 non-null    float64 
 7   deliveries                131 non-null    float64 
 8   menarche_age              131 non-null    float64 
 9   cycle_days                131 non-null    float64 
 10  cycle_interval            131 non-null    float64 
 11  cycle_disorder            131 non-null    float64 
 12  sexual_life_start         131 non-null    float64 
 13  partner_age               131 non-null    float64 

In [1446]:
df_preg.to_excel('FINAL_pregnancy.xlsx', index=False)

In [1447]:
df_preg.shape

(131, 32)

In [1448]:
print("=" * 60)
print("БАЛАНС КЛАССОВ В ДАТАСЕТАХ")
print("=" * 60)

# Pregnancy
print("\n[ PREGNANCY ]")
print(df_preg['pregnancy'].value_counts())
print("Доля классов:")
print(df_preg['pregnancy'].value_counts(normalize=True).round(4))

# Delivery
print("\n[ DELIVERY ]")
print(df_main['delivery'].value_counts())
print("Доля классов:")
print(df_main['delivery'].value_counts(normalize=True).round(4))

# Соотношение положительного класса
print("\n[ СООТНОШЕНИЕ positive/negative ]")
preg_pos = df_preg['pregnancy'].mean()
del_pos = df_main['delivery'].mean()
print(f"Pregnancy positive class: {preg_pos:.3f} (соотношение 1:{(1-preg_pos)/preg_pos:.1f})")
print(f"Delivery positive class:  {del_pos:.3f} (соотношение 1:{(1-del_pos)/del_pos:.1f})")

БАЛАНС КЛАССОВ В ДАТАСЕТАХ

[ PREGNANCY ]
pregnancy
1.0    116
0.0     15
Name: count, dtype: int64
Доля классов:
pregnancy
1.0    0.8855
0.0    0.1145
Name: proportion, dtype: float64

[ DELIVERY ]
delivery
0.0    108
1.0     87
Name: count, dtype: int64
Доля классов:
delivery
0.0    0.5538
1.0    0.4462
Name: proportion, dtype: float64

[ СООТНОШЕНИЕ positive/negative ]
Pregnancy positive class: 0.885 (соотношение 1:0.1)
Delivery positive class:  0.446 (соотношение 1:1.2)
